In [1]:
import os
import warnings
from pathlib import Path
from typing import Any

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from model_config import LocalModel, RemoteModel  # noqa: E402
from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = RemoteModel.GPT_OSS_120B.value
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0.value

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

In [5]:
from uuid import uuid4

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

### Load Data

In [6]:
fp: str = "../../data/chelsea_transfer_news.pdf"
loader = PyPDFLoader(fp)
docs = loader.load()


# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=100
)
splits = text_splitter.split_documents(docs)

console.print(f"Number of chunks: {len(splits)}", style="info")

Number of chunks: 13

### Indexing

In [7]:
collection_name: str = "cfc_transfer_news"

emb_model = OllamaEmbeddings(
    model=LocalModel.MXBAI_EMBED_LARGE.value,
)
emb = emb_model.embed_documents("Hello world")
emb_size: int = len(emb[0])
console.print(f"Embedding size: {emb_size}", style="info")

client = QdrantClient(url=settings.QDRANT_URL)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Vector store
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=emb_model,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Embedding size: 1024

## Query Transformation

- Query transformations are a set of approaches focused on re-writing and / or modifying questions for retrieval.

In [8]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template: str = """
<system>
You are an AI language model assistant. 
<role>
Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector 
database.
</role>
<instructions>
By generating multiple perspectives on the user question, your goal is to help the user overcome some of the 
limitations of the distance-based similarity search. Provide these alternative questions separated by newlines. 
<original_question>
{question}
</original_question>
</instructions>
</system>
"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives | local_llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [9]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list[Any]]) -> list[Any]:
    """
    Get the unique union of lists of documents.

    Parameters
    ----------
    documents : list[list[Any]]
        A list of lists containing documents.

    Returns
    -------
    list[Any]
        A list containing unique documents.
    """
    # Flatten list of lists and convert each Document to a string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))

    return [loads(doc) for doc in unique_docs]


# Retrieve
question: str = "What players are likely to exit Chelsea?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
retr_docs = retrieval_chain.invoke({"question": question})
len(retr_docs)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_49892/1780963150.py:22: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


5

In [10]:
console.print(retr_docs)

[
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 10,
            'page_label': '11',
            '_id': '3face695-106e-4fd7-bc99-84312d7e0369',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content="Any further signings at Chelsea are dependent on exits.\nChelsea have held also held talks 
over the signing of\xa0Xavi Simons\xa0at\xa0RB Leipzig.\n23 Jul\n08:08\nRB Leipzig looking at Chelsea's 
Chukwuemeka\nLatest from Sky in Germany's Florian Plettenberg and\xa0Philipp Heinze:\nRB Leipzig\xa0are pursuing a 
move for\xa0Chelsea\xa0midfielder\xa0Carney Chukwuemeka,\naccording to\xa0Sky in Germany.\xa0\nChukwuemeka spent 
the\xa0second half of last season on loan at\xa0Borussia\nDortmund.\xa0\nBut it is Leipzig that are now pushing to 
sign the 21-year-old, who has three\nyears left on his deal.\nSky Sports News has reported that Chelsea are looking
at Leipzig's attacking\nmidfielder\xa0Xavi Simons.\xa0\nChelsea have other targets as well, and any more incomings 
are dependent on\nplayers also leaving.\xa0\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live 
updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and… 11/17"
    ),
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 1,
            'page_label': '2',
            '_id': 'd982eea8-139f-4770-963b-95d79387b53f',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content="Pinned\n4h ago\n14:39\n'Whites express interest in signing Chelsea duo'\nFulham\xa0have 
expressed an interest in signing both\xa0Raheem\nSterling\xa0and\xa0Kiernan 
Dewsbury-Hall\xa0from\xa0Chelsea\xa0this summer, according to\nthe\xa0BBC.\xa0\nIt is suggested that Sterling, 30, 
who spent last season on loan at Arsenal\nafter being frozen out by Blues boss Enzo Maresca, has no future at 
Stamford\nBridge and is among a 'bomb squad' of players surplus to requirements.\xa0\nDewsbury-Hall, meanwhile, 
found minutes hard to come by after signing from\nLeicester last summer, although it is reported he is viewed as a 
valued\nmember of Chelsea's squad.\xa0\nChelsea reportedly prefer to sell the players who were sent on loan and 
not\nconsidered in the manager's plans next summer, which is true of Sterling's\ncase.\xa0\nSterling has two years 
remaining on his contract at Stamford Bridge.\xa0\n24 Jul\n18:28\nChelsea target Hato not in Ajax 
squad\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, 
loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 2/17"
    ),
    Doc

In [11]:
from operator import itemgetter

# RAG Prompt
rag_template: str = """
<instructions>
Answer the following question based on this context:
<context>{context}</context>
<question>{question}</question
</instructions
"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)
final_rag_chain = (
    {
        "context": retrieval_chain,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | local_llm
    | StrOutputParser()
)

In [12]:
response = final_rag_chain.invoke({"question": question})
console.print(response)

The players likely to exit Chelsea this summer, according to the provided context, include:

1. Joao Felix (Portuguese playmaker)
2. Raheem Sterling (winger)
3. Ben Chilwell (England left-back)
4. Renato Veiga (on loan at Juventus)
5. Axel Disasi (defender, on loan with Aston Villa)
6. Carney Chukwuemeka (midfielder, RB Leipzig are pursuing a move for him)
7. Christopher Nkunku (Inter Milan are interested in him)